# Tutorial_yolov5_voc.ipynb

## 1. Library Install& Import

In [13]:
import torch
import torchvision
from torchvision import datasets
import os
import sys

print("Python version :", sys.version)
print("PyTorch version :", torch.__version__)
print("Torchvision version :", torchvision.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using Device :", device)

Python version : 3.10.0 (default, Mar  3 2022, 09:58:08) [GCC 7.5.0]
PyTorch version : 2.6.0+cu118
Torchvision version : 0.21.0+cu118
Using Device : cuda


## 2. Download Pascal VOC Dataset (torchvision)

In [14]:
from torchvision.datasets import VOCDetection
# Specify the root path to use locally (or on Colab)
# Example: VOCdevkit folder will be created under the current directory ('.')
DATA_ROOT = "./"
# Download the train dataset using VOCDetection
voc_train = VOCDetection(
    root=DATA_ROOT,
    year="2007",
    image_set="train",  # trainval, test, etc. available
    download=True,       # automatic download
    transform=None,
    target_transform=None
)
print("VOC 2007 trainset download completed!")
print("Number of data:", len(voc_train))
print("Check saved path structure:", os.listdir(os.path.join(DATA_ROOT, "VOCdevkit")))

VOC 2007 trainset download completed!
Number of data: 2501
Check saved path structure: ['VOC2007']


## 3. Function to convert VOC XML into YOLO text

In [15]:
import os
import xml.etree.ElementTree as ET
import glob
import random
import shutil
import numpy as np
# Path settings - modified VOCdevkit path
voc_root = "/userHome/userhome1/chaewoon/VOCdevkit/VOC2007"
voc_annotations = os.path.join(voc_root, "Annotations")
voc_images = os.path.join(voc_root, "JPEGImages")
# YOLO dataset save location - saved in the same user path to solve permission issues
yolo_dataset = "/userHome/userhome1/chaewoon/yolo_voc_dataset"
os.makedirs(yolo_dataset, exist_ok=True)
# Class list
voc_classes = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]
# Function to convert VOC XML to YOLO text
def convert_voc_to_yolo(xml_file, output_dir):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    size = root.find("size")
    width = int(size.find("width").text)
    height = int(size.find("height").text)
    
    filename = root.find("filename").text
    base_name = os.path.splitext(filename)[0]
    
    yolo_lines = []
    
    for obj in root.findall("object"):
        # Get class name and ID
        class_name = obj.find("name").text
        if class_name not in voc_classes:
            continue
            
        class_id = voc_classes.index(class_name)
        
        # Check difficult flag (optional)
        difficult = obj.find("difficult")
        if difficult is not None and int(difficult.text) == 1:
            continue
        
        # Get bounding box coordinates
        bbox = obj.find("bndbox")
        xmin = float(bbox.find("xmin").text)
        ymin = float(bbox.find("ymin").text)
        xmax = float(bbox.find("xmax").text)
        ymax = float(bbox.find("ymax").text)
        
        # Convert to YOLO format (center x, y, width, height) - all values between 0~1
        x_center = ((xmin + xmax) / 2) / width
        y_center = ((ymin + ymax) / 2) / height
        box_width = (xmax - xmin) / width
        box_height = (ymax - ymin) / height
        
        # Save in YOLO format
        yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}")
    
    # Save converted labels
    if yolo_lines:  # Only save if there are valid objects
        with open(os.path.join(output_dir, f"{base_name}.txt"), "w") as f:
            f.write("\n".join(yolo_lines))
        return True
    return False

## 4. Dataset configuration (train/val segmentation) & Create a data.yaml file

In [16]:
def setup_yolo_dataset():
    # Create directories
    os.makedirs(os.path.join(yolo_dataset, "images", "train"), exist_ok=True)
    os.makedirs(os.path.join(yolo_dataset, "images", "val"), exist_ok=True)
    os.makedirs(os.path.join(yolo_dataset, "images", "test"), exist_ok=True)  # 테스트 디렉토리 추가
    os.makedirs(os.path.join(yolo_dataset, "labels", "train"), exist_ok=True)
    os.makedirs(os.path.join(yolo_dataset, "labels", "val"), exist_ok=True)
    os.makedirs(os.path.join(yolo_dataset, "labels", "test"), exist_ok=True)  # 테스트 디렉토리 추가
    
    # YOLO label save directory
    temp_labels_dir = os.path.join(yolo_dataset, "temp_labels")
    os.makedirs(temp_labels_dir, exist_ok=True)
    
    # Label conversion for all XML files
    xml_files = glob.glob(os.path.join(voc_annotations, "*.xml"))
    valid_images = []
    
    for xml_file in xml_files:
        base_name = os.path.splitext(os.path.basename(xml_file))[0]
        img_file = os.path.join(voc_images, f"{base_name}.jpg")
        
        # Only valid if image file exists and label conversion is successful
        if os.path.exists(img_file) and convert_voc_to_yolo(xml_file, temp_labels_dir):
            valid_images.append(base_name)
    
    print(f"Found {len(valid_images)} valid images and labels.")
    
    # train/val/test split (60/20/20)
    random.shuffle(valid_images)
    train_split = int(len(valid_images) * 0.6)
    val_split = int(len(valid_images) * 0.8)
    
    train_images = valid_images[:train_split]
    val_images = valid_images[train_split:val_split]
    test_images = valid_images[val_split:]
    
    print(f"Training: {len(train_images)} images ({len(train_images)/len(valid_images)*100:.1f}%)")
    print(f"Validation: {len(val_images)} images ({len(val_images)/len(valid_images)*100:.1f}%)")
    print(f"Testing: {len(test_images)} images ({len(test_images)/len(valid_images)*100:.1f}%)")
    
    # Copy image and label files
    for img_set, subset in [(train_images, "train"), (val_images, "val"), (test_images, "test")]:
        for img_name in img_set:
            # Copy image
            src_img = os.path.join(voc_images, f"{img_name}.jpg")
            dst_img = os.path.join(yolo_dataset, "images", subset, f"{img_name}.jpg")
            shutil.copy(src_img, dst_img)
            
            # Copy label
            src_label = os.path.join(temp_labels_dir, f"{img_name}.txt")
            dst_label = os.path.join(yolo_dataset, "labels", subset, f"{img_name}.txt")
            if os.path.exists(src_label):
                shutil.copy(src_label, dst_label)
    
    # Delete temporary label directory
    # shutil.rmtree(temp_labels_dir)
    
    # Create dataset.yaml file for YOLOv5 training
    yaml_content = {
        'path': os.path.abspath(yolo_dataset),
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'nc': len(voc_classes),
        'names': voc_classes
    }
    
    with open(os.path.join(yolo_dataset, 'dataset.yaml'), 'w') as f:
        yaml.dump(yaml_content, f, default_flow_style=False)
    
    print(f"Created dataset.yaml in {yolo_dataset}")
    
    return train_images, val_images, test_images

## 5. Training YOLOv5 (Ultralytics)

In [17]:
from ultralytics import YOLO
import os
import torch
# Load model
model = YOLO("yolov5s.pt")
print("YOLOv5 model loading complete!")
# Training - using GPU
model.train(
    data=os.path.join(yolo_dataset, "data.yaml"),
    epochs=5,
    batch=4,
    imgsz=416,
    name="yolov5_voc_demo",
    device=0,
    workers=0    
)
# 예: best_model.pt로 저장
torch.save(model.state_dict(), 'best_model.pt')

PRO TIP 💡 Replace 'model=yolov5s.pt' with new 'model=yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.

YOLOv5 model loading complete!
Ultralytics 8.3.96 🚀 Python-3.10.0 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3090, 24253MiB)
engine/trainer: task=detect, mode=train, model=yolov5s.pt, data=/userHome/userhome1/chaewoon/yolo_voc_dataset/data.yaml, epochs=5, time=None, patience=100, batch=4, imgsz=416, save=True, save_period=-1, cache=False, device=0, workers=0, project=None, name=yolov5_voc_demo5, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, co

train: Scanning /userHome/userhome1/chaewoon/yolo_voc_dataset/labels/train.cache... 4846 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4846/4846 [00:00<?, ?it/s]
val: Scanning /userHome/userhome1/chaewoon/yolo_voc_dataset/labels/val.cache... 2438 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2438/2438 [00:00<?, ?it/s]

Plotting labels to runs/detect/yolov5_voc_demo5/labels.jpg... 


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000417, momentum=0.9) with parameter groups 69 weight(decay=0.0), 76 weight(decay=0.0005), 75 bias(decay=0.0)
Image sizes 416 train, 416 val
Using 0 dataloader workers
Logging results to runs/detect/yolov5_voc_demo5
Starting training for 5 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


        1/5     0.615G      1.125      1.913      1.258         14        416: 100%|██████████| 1212/1212 [02:17<00:00,  8.82it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 305/305 [00:22<00:00, 13.81it/s]


...

        5/5      0.83G      1.047      1.279      1.218         14        416: 100%|██████████| 1212/1212 [03:07<00:00,  6.47it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 305/305 [00:41<00:00,  7.29it/s]

                   all       2438       6198        0.8      0.777      0.842      0.628



5 epochs completed in 0.280 hours.
Optimizer stripped from runs/detect/yolov5_voc_demo5/weights/last.pt, 18.5MB
Optimizer stripped from runs/detect/yolov5_voc_demo5/weights/best.pt, 18.5MB

Validating runs/detect/yolov5_voc_demo5/weights/best.pt...
Ultralytics 8.3.96 🚀 Python-3.10.0 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3090, 24253MiB)
YOLOv5s summary (fused): 84 layers, 9,119,276 parameters, 0 gradients, 23.9 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 305/305 [00:49<00:00,  6.11it/s]


                   all       2438       6198        0.8      0.775      0.842      0.627
             aeroplane        111        141      0.912      0.806       0.93      0.715
               bicycle        112        156       0.82      0.827      0.874      0.637
                  bird        163        248      0.783      0.773      0.827      0.615
                  boat         92        150      0.637        0.7      0.732      0.468
                bottle        119        231       0.69      0.654      0.659      0.456
                   bus        101        122      0.756       0.91      0.911      0.767
                   car        349        596      0.878      0.857      0.926      0.725
                   cat        151        166      0.909      0.785      0.891      0.728
                 chair        214        406      0.808       0.67      0.779      0.549
                   cow         74        139      0.711      0.806      0.819      0.586
           diningtabl

## 6. Validation and inference tests

In [18]:
# Load the best.pt model from training results and run validation
results = model.val()
print(results)
# Inference test
import cv2
import matplotlib.pyplot as plt
# Set test image path
# Manually specify image ID
image_dir = "/userHome/userhome1/chaewoon/VOCdevkit/VOC2007/JPEGImages"
# Get list of image files from the image directory
import glob
image_files = glob.glob(os.path.join(image_dir, "*.jpg"))
if not image_files:
    print(f"No image files in the path: {image_dir}")
else:
    # Use the first image file
    sample_img_path = image_files[0]
    sample_img_id = os.path.splitext(os.path.basename(sample_img_path))[0]
    
    print(f"Image to test: {sample_img_path}")
    
    # Perform prediction with the model
    preds = model.predict(source=sample_img_path, conf=0.25, save=True)
    output_img_path = preds[0].path
    
    # Visualization
    img_result = cv2.imread(output_img_path)
    img_result = cv2.cvtColor(img_result, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(8,8))
    plt.imshow(img_result)
    plt.axis("off")
    plt.title("YOLOv5 Inference on Pascal VOC")
    plt.show()

Ultralytics 8.3.96 🚀 Python-3.10.0 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3090, 24253MiB)
YOLOv5s summary (fused): 84 layers, 9,119,276 parameters, 0 gradients, 23.9 GFLOPs


val: Scanning /userHome/userhome1/chaewoon/yolo_voc_dataset/labels/val.cache... 2438 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2438/2438 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 610/610 [00:53<00:00, 11.35it/s]


                   all       2438       6198      0.801      0.776      0.842      0.628
             aeroplane        111        141      0.912      0.805      0.931      0.717
               bicycle        112        156       0.82      0.827      0.874      0.636
                  bird        163        248      0.782      0.768      0.828      0.616
                  boat         92        150       0.64      0.707      0.732      0.467
                bottle        119        231      0.686      0.654      0.659      0.455
                   bus        101        122      0.758       0.91      0.911      0.768
                   car        349        596      0.877      0.857      0.926      0.726
                   cat        151        166      0.909      0.785      0.891      0.727
                 chair        214        406      0.808      0.672      0.779       0.55
                   cow         74        139       0.71      0.806       0.82      0.588
           diningtabl

<Figure size 800x800 with 1 Axes>